# Legal NLP Dissertation — Experiment Notebook
**Topic**: Legal Document Summarisation using NLP Techniques for Indian Law

**Pipeline**: Indian Kanoon/IndiaCode → Preprocessing → Legal-BERT RAG → LED → ROUGE/BERTScore

---

## 1. Setup & Imports

In [ ]:
import sys
sys.path.insert(0, '..')   # add project root to path

import json
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams.update({'figure.dpi': 120, 'font.size': 11})

print('Setup complete')

## 2. Data Collection

We collect judgments from Indian Kanoon and Acts from IndiaCode.

In [ ]:
from data_pipeline.scraper import collect_corpus

corpus = collect_corpus(
    ik_queries=['Supreme Court contract 2022',
                'RERA judgment 2023',
                'breach of contract India'],
    act_titles=['Indian Contract Act',
                'RERA', 'IPC'],
    max_per_source=20,
)

print(f'Total documents: {len(corpus)}')
print('\nDocument types:')
pd.Series([d['doc_type'] for d in corpus]).value_counts()

## 3. Preprocessing

In [ ]:
from data_pipeline.preprocessor import LegalPreprocessor

preprocessor   = LegalPreprocessor(bert_chunk_size=400, bert_overlap=80)
processed_docs = preprocessor.process_batch(corpus)

# Statistics
chunk_counts = [len(d['bert_chunks']) for d in processed_docs]
token_counts = [c.tokens for d in processed_docs for c in d['bert_chunks']]

print(f'Total BERT chunks: {sum(chunk_counts)}')
print(f'Mean chunk tokens: {sum(token_counts)/len(token_counts):.1f}')
print(f'Max chunk tokens : {max(token_counts)}')

In [ ]:
# Plot chunk token distribution
plt.figure(figsize=(8, 3))
plt.hist(token_counts, bins=30, color='steelblue', edgecolor='white', alpha=0.9)
plt.axvline(512, color='red', linestyle='--', label='BERT limit (512)')
plt.xlabel('Tokens per chunk')
plt.ylabel('Count')
plt.title('Chunk Token Distribution')
plt.legend()
plt.tight_layout()
plt.savefig('../figures/chunk_distribution.png', dpi=150)
plt.show()

## 4. Build FAISS Index with Legal-BERT

In [ ]:
from models.embedder import LegalBERTEmbedder, FAISSVectorStore

embedder = LegalBERTEmbedder()                   # loads Legal-BERT
store    = FAISSVectorStore(embedder)
store.add_chunks(processed_docs)                  # embed + index
store.save('../faiss_index')

print(f'Index built: {store.index.ntotal} vectors')

## 5. Test Retrieval

In [ ]:
query   = 'breach of contract damages Supreme Court'
results = store.search(query, top_k=5)

print(f'Query: {query}\n')
for i, r in enumerate(results, 1):
    title = r.get('metadata', {}).get('title', r['doc_id'])
    print(f'{i}. [{r["score"]:.4f}] {title}')
    print(f'   {r["text"][:200]}...')
    print()

## 6. Summarisation with LED

In [ ]:
from models.summarizer   import LEDSummarizer
from rag_engine.pipeline import RAGPipeline

summarizer = LEDSummarizer()                     # loads LED-base-16384

pipeline = RAGPipeline(
    vector_store = store,
    summarizer   = summarizer,
    initial_k    = 15,
    final_k      = 6,
)

result = pipeline.run('breach of contract remedies')

print('SUMMARY:')
print('='*60)
print(result['summary'])
print('\nBased on', len(result['sources']), 'source chunks')

## 7. Evaluation vs Baselines

In [ ]:
from evaluation.metrics import Evaluator

# ── Replace these with your actual test set ────────────────────────────────
# Format: each item = (full document text, gold summary)
# Gold summaries can come from SCI headnotes or manually written summaries
test_documents = [
    corpus[0]['text'],
    corpus[1]['text'],
]
gold_summaries = [
    "The court held that breach of contract was established and awarded damages.",   # replace!
    "The arbitration clause was upheld and parties were directed to arbitrate.",    # replace!
]

ev = Evaluator()

# Our system
our_preds   = [pipeline.summarise_document(d)['summary'] for d in test_documents]
our_result  = ev.evaluate(our_preds, gold_summaries, model_name='LED+RAG (ours)')

# Baselines
bart_preds  = ev.generate_baseline_summaries(test_documents, model='bart')
bart_result = ev.evaluate(bart_preds, gold_summaries, model_name='BART-base')

t5_preds    = ev.generate_baseline_summaries(test_documents, model='t5')
t5_result   = ev.evaluate(t5_preds, gold_summaries, model_name='T5-small')

ext_preds   = ev.generate_baseline_summaries(test_documents, model='extractive')
ext_result  = ev.evaluate(ext_preds, gold_summaries, model_name='Lead-5')

all_results = [ext_result, t5_result, bart_result, our_result]
ev.compare(all_results)
ev.save_results(all_results, '../evaluation/results.json')

In [ ]:
# ── Plot ROUGE comparison ──────────────────────────────────────────────────
import numpy as np

models   = [r.model_name for r in all_results]
rouge1   = [r.rouge1   for r in all_results]
rouge2   = [r.rouge2   for r in all_results]
rougeL   = [r.rougeL   for r in all_results]
bert     = [r.bertscore_f1 for r in all_results]

x    = np.arange(len(models))
w    = 0.2
fig, ax = plt.subplots(figsize=(10, 4))
ax.bar(x - 1.5*w, rouge1, w, label='ROUGE-1', color='#4e79a7')
ax.bar(x - 0.5*w, rouge2, w, label='ROUGE-2', color='#f28e2b')
ax.bar(x + 0.5*w, rougeL, w, label='ROUGE-L', color='#59a14f')
ax.bar(x + 1.5*w, bert,   w, label='BERTScore', color='#e15759')
ax.set_xticks(x)
ax.set_xticklabels(models, rotation=15, ha='right')
ax.set_ylabel('Score')
ax.set_title('Model Comparison — Indian Legal Summarisation')
ax.legend()
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig('../figures/model_comparison.png', dpi=150)
plt.show()

## 8. Fine-tuning LED (optional — do on GPU)

In [ ]:
# Only run this if you have:
#   - A GPU with 16+ GB VRAM
#   - Annotated (document, summary) training pairs

# from models.summarizer import LEDFineTuner

# train_data = [
#     {"document": full_judgment_text, "summary": headnote_or_annotated_summary},
#     ...  # need at least 100–500 pairs for meaningful fine-tuning
# ]
# eval_data = train_data[-20:]   # hold-out validation set

# tuner = LEDFineTuner(output_dir='../finetuned_led')
# tuner.train(train_data, eval_data, epochs=3)

print('Fine-tuning disabled — uncomment above when GPU available')